# SVR Momentum Grid Search

Vectorized grid search over `svr_momentum_signal_optimized` parameters to find the
combination that maximises Sharpe ratio on `USD-SOFR-1D-Q12STIRT` / `IMM_4xIMM_5`.

Uses the optimized signal function (~300x faster than the base version) so the
full grid can be swept in reasonable time.

**Grid dimensions:**
- `svr_c` — regularization strength
- `svr_gamma` — RBF kernel bandwidth
- `signal_threshold` — minimum predicted return to enter a trade
- `retrain_every` — retraining frequency (bars)
- `ewma_short_spans` / `ewma_long_spans` — feature window presets

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import itertools
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import seaborn as sns

sys.path.append("../../")

from BT.signals import svr_momentum_signal_optimized
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

NYC = pytz.timezone("America/New_York")

In [ ]:
# ── Load data (same as the single-run notebook) ──────────────────────────
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
ts_builder = TimeseriesBuilder()
trade_bpv = 100_000.0

start = NYC.localize(datetime.datetime(2026, 1, 4, 18, 0))
end = NYC.localize(datetime.datetime(2026, 3, 27, 17, 0))

q = UnifiedQuery(
    curve="USD-SOFR-1D-Q12STIRT",
    tenor="IMM_4xIMM_5",
    value=UnifiedValue.IRS_RATE,
)

intraday_df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q],
    freq="1min",
    mdps={"IRS": curve_mdp},
    ignore_cache_miss=True,
)
intraday_df = intraday_df.sort_index()

rate_series = pd.to_numeric(intraday_df.iloc[:, 0], errors="coerce").dropna()
rate_series = rate_series.resample("30min").last().dropna()
rate_series.name = "IMM_4xIMM_5"

print(f"Loaded {len(rate_series)} bars from {rate_series.index[0]} to {rate_series.index[-1]}")

In [ ]:
# ── Vectorized PnL helper ────────────────────────────────────────────────
def vectorized_metrics(signal_result, rate_series, trade_bpv=100_000.0, cost_bps=0.25):
    """Quick vectorized PnL, Sharpe, drawdown, and trade count from a signal result."""
    position = signal_result.execution_position.fillna(0.0)
    delta = rate_series.diff().fillna(0.0)
    gross_pnl = -position * delta * trade_bpv * 100.0

    # Transaction costs: each position change costs cost_bps per side
    trade_sides = position.diff().abs()
    trade_sides.iloc[0] = position.iloc[0].abs() if hasattr(position.iloc[0], "abs") else abs(position.iloc[0])
    costs = trade_sides * cost_bps * trade_bpv
    net_pnl = gross_pnl - costs

    cumulative = net_pnl.cumsum()
    total_pnl = float(cumulative.iloc[-1]) if len(cumulative) else 0.0
    n_trades = float(trade_sides.sum())

    # Annualized Sharpe
    std = float(net_pnl.std())
    bars_per_day = 48  # 30-min bars
    ann_factor = bars_per_day * 252
    sharpe = float(net_pnl.mean() / std * np.sqrt(ann_factor)) if std > 0 else 0.0

    # Max drawdown
    drawdown = cumulative - cumulative.cummax()
    max_dd = float(drawdown.min()) if len(drawdown) else 0.0

    # Win rate
    nonzero = net_pnl[net_pnl != 0.0]
    win_rate = float((nonzero > 0).mean()) if len(nonzero) else 0.0

    return {
        "total_pnl": total_pnl,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "n_trades": n_trades,
        "win_rate": win_rate,
        "cumulative": cumulative,
    }

In [ ]:
# ── Define grid ──────────────────────────────────────────────────────────
# Feature window presets (short_spans, long_spans, label)
WINDOW_PRESETS = {
    "matlab_hourly": ([24, 48, 120, 240], [360, 1080, 1440]),
    "fast":          ([12, 24, 48],        [120, 360, 720]),
    "slow":          ([48, 120, 240, 480], [720, 1440, 2880]),
}

GRID = {
    "svr_c":             [10.0, 100.0, 400.0, 1000.0],
    "svr_gamma":         [0.01, 0.1, 0.5, 1.0],
    "signal_threshold":  [0.0, 0.0001, 0.0005],
    "retrain_every":     [24, 48, 96],
    "windows":           list(WINDOW_PRESETS.keys()),
}

n_combos = 1
for v in GRID.values():
    n_combos *= len(v)
print(f"Grid has {n_combos} parameter combinations")

In [ ]:
# ── Run grid search ──────────────────────────────────────────────────────
results = []
keys = list(GRID.keys())
combos = list(itertools.product(*GRID.values()))

t_start = time.perf_counter()
for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    window_key = params.pop("windows")
    short_spans, long_spans = WINDOW_PRESETS[window_key]

    try:
        sig = svr_momentum_signal_optimized(
            rate_series,
            ewma_short_spans=short_spans,
            ewma_long_spans=long_spans,
            svr_c=params["svr_c"],
            svr_gamma=params["svr_gamma"],
            svr_epsilon=0.000000025,
            signal_threshold=params["signal_threshold"],
            train_fraction=0.7,
            retrain_every=params["retrain_every"],
        )
        m = vectorized_metrics(sig, rate_series, trade_bpv=trade_bpv)
        row = {
            "windows": window_key,
            **params,
            "total_pnl": m["total_pnl"],
            "sharpe": m["sharpe"],
            "max_drawdown": m["max_drawdown"],
            "n_trades": m["n_trades"],
            "win_rate": m["win_rate"],
        }
    except Exception as exc:
        row = {
            "windows": window_key,
            **params,
            "total_pnl": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "n_trades": np.nan,
            "win_rate": np.nan,
        }

    results.append(row)

    if (i + 1) % 50 == 0 or (i + 1) == len(combos):
        elapsed = time.perf_counter() - t_start
        rate = (i + 1) / elapsed
        eta = (len(combos) - i - 1) / rate if rate > 0 else 0
        print(f"  [{i+1}/{len(combos)}] {elapsed:.0f}s elapsed, {rate:.1f} combos/s, ETA {eta:.0f}s")

grid_df = pd.DataFrame(results)
print(f"\nCompleted {len(grid_df)} combinations in {time.perf_counter() - t_start:.1f}s")
grid_df.head()

In [ ]:
# ── Top results by Sharpe ────────────────────────────────────────────────
top = grid_df.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False).head(20)
display(top.style.format({
    "total_pnl": "${:,.0f}",
    "sharpe": "{:.2f}",
    "max_drawdown": "${:,.0f}",
    "n_trades": "{:.0f}",
    "win_rate": "{:.1%}",
}).background_gradient(subset=["sharpe"], cmap="RdYlGn"))

In [ ]:
# ── Heatmap: Sharpe by C × gamma (averaged over other params) ─────────
pivot = grid_df.pivot_table(
    values="sharpe",
    index="svr_c",
    columns="svr_gamma",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0, ax=ax)
ax.set_title("Mean Sharpe: C × γ (averaged over windows, threshold, retrain)")
ax.set_ylabel("svr_c")
ax.set_xlabel("svr_gamma")
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: Sharpe by windows × retrain_every ───────────────────────────
pivot2 = grid_df.pivot_table(
    values="sharpe",
    index="windows",
    columns="retrain_every",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot2, annot=True, fmt=".2f", cmap="RdYlGn", center=0, ax=ax)
ax.set_title("Mean Sharpe: Window Preset × Retrain Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# ── Sharpe by signal_threshold ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
grid_df.boxplot(column="sharpe", by="signal_threshold", ax=ax)
ax.set_title("Sharpe Distribution by Signal Threshold")
ax.set_xlabel("signal_threshold")
ax.set_ylabel("Sharpe")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# ── Trade count vs Sharpe scatter ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    grid_df["n_trades"],
    grid_df["sharpe"],
    c=grid_df["total_pnl"],
    cmap="RdYlGn",
    alpha=0.6,
    s=20,
)
plt.colorbar(scatter, label="Total PnL")
ax.set_xlabel("Number of Trade Sides")
ax.set_ylabel("Sharpe Ratio")
ax.set_title("Trade Frequency vs Sharpe (color = Total PnL)")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Run the best combo and plot its equity curve ─────────────────────────
best = grid_df.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False).iloc[0]
print("Best parameters:")
for col in ["windows", "svr_c", "svr_gamma", "signal_threshold", "retrain_every"]:
    print(f"  {col}: {best[col]}")
print(f"  Sharpe: {best['sharpe']:.2f}")
print(f"  Total PnL: ${best['total_pnl']:,.0f}")
print(f"  Max Drawdown: ${best['max_drawdown']:,.0f}")
print(f"  Trades: {best['n_trades']:.0f}")
print(f"  Win Rate: {best['win_rate']:.1%}")

best_short, best_long = WINDOW_PRESETS[best["windows"]]
best_sig = svr_momentum_signal_optimized(
    rate_series,
    ewma_short_spans=best_short,
    ewma_long_spans=best_long,
    svr_c=best["svr_c"],
    svr_gamma=best["svr_gamma"],
    svr_epsilon=0.000000025,
    signal_threshold=best["signal_threshold"],
    train_fraction=0.7,
    retrain_every=int(best["retrain_every"]),
)
best_metrics = vectorized_metrics(best_sig, rate_series, trade_bpv=trade_bpv)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
rate_series.plot(ax=axes[0], title="IMM_4xIMM_5 Rate")
best_sig.execution_position.fillna(0.0).plot(ax=axes[1], title="Best Execution Position", color="black")
best_metrics["cumulative"].plot(ax=axes[2], title=f"Best Equity Curve (Sharpe={best['sharpe']:.2f})", color="tab:green")
axes[2].set_ylabel("Cumulative PnL")
plt.tight_layout()
plt.show()

In [ ]:
# ── Compare top 5 equity curves ─────────────────────────────────────────
top5 = grid_df.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False).head(5)

fig, ax = plt.subplots(figsize=(14, 5))
for rank, (_, row) in enumerate(top5.iterrows(), 1):
    short, long = WINDOW_PRESETS[row["windows"]]
    sig = svr_momentum_signal_optimized(
        rate_series,
        ewma_short_spans=short,
        ewma_long_spans=long,
        svr_c=row["svr_c"],
        svr_gamma=row["svr_gamma"],
        svr_epsilon=0.000000025,
        signal_threshold=row["signal_threshold"],
        train_fraction=0.7,
        retrain_every=int(row["retrain_every"]),
    )
    m = vectorized_metrics(sig, rate_series, trade_bpv=trade_bpv)
    label = f"#{rank} C={row['svr_c']:.0f} γ={row['svr_gamma']} thr={row['signal_threshold']} rt={int(row['retrain_every'])} [{row['windows']}] S={row['sharpe']:.2f}"
    m["cumulative"].plot(ax=ax, label=label, alpha=0.8)

ax.set_title("Top 5 Parameter Combinations — Equity Curves")
ax.set_ylabel("Cumulative PnL")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# ── Export full grid to CSV for further analysis ──────────────────────────
grid_df.to_csv("svr_momentum_grid_results.csv", index=False)
print(f"Saved {len(grid_df)} rows to svr_momentum_grid_results.csv")